In [ ]:
import numpy as np
from datetime import date
from matplotlib import pyplot as plt
import marineHeatWaves as mhw
import xarray as xr
from cmethods import adjust
import pandas as pd

In [ ]:
obsh = xr.open_dataset("path/OISST_daily_mean_1982_2022.nc")
sst = obsh.sst.sel(time=slice('1984','2017'))
sst.shape

In [ ]:
obsh_sst = sst.sel(time=~((sst.time.dt.month == 2) & (sst.time.dt.day == 29)))
obsh_sst.shape

In [ ]:
simh = xr.open_dataset("path/merged_CanESM5_sst_day_1983_2016_ensmean_LY1_rg1by4.nc")
simp = xr.open_dataset("path/merged_CanESM5_sst_day_1983_2016_ensmean_LY1_rg1by4.nc")

In [ ]:
simh_sst = xr.DataArray(simh.tos, coords=[obsh_sst.time, obsh_sst.lat, obsh_sst.lon], dims=['time','lat','lon'], name='sst')
simp_sst = xr.DataArray(simp.tos, coords=[obsh_sst.time, obsh_sst.lat, obsh_sst.lon], dims=['time','lat','lon'], name='sst')

In [ ]:
simp_sst.shape

In [ ]:
obsh_sst_grouped = obsh_sst.groupby("time.dayofyear")
simh_sst_grouped = simh_sst.groupby("time.dayofyear")
simp_sst_grouped = simp_sst.groupby("time.dayofyear")

In [ ]:
if not (obsh_sst_grouped.groups.keys() == simh_sst_grouped.groups.keys() == simp_sst_grouped.groups.keys()):
    raise ValueError("The datasets have inconsistent time periods or DOY groups!")

In [ ]:
output_dir = "../output_data"

for doy, indices in obsh_sst_grouped.groups.items():
    print(f"Processing Day of Year: {doy}")

    obsh_doy = obsh_sst.isel(time=indices)
    simh_doy = simh_sst.isel(time=indices)
    simp_doy = simp_sst.isel(time=indices)

    qm_adjusted = adjust(
        method="quantile_mapping",
        obs=obsh_doy,
        simh=simh_doy,
        simp=simp_doy,
        n_quantiles=250,
        kind="+",
    )

    qm_adjusted_xr = qm_adjusted.sst

    output_file = f"{output_dir}/qm_adjusted_doy_{doy:03d}.nc"
    qm_adjusted_xr.to_netcdf(output_file)

    print(f"Saved corrected data for DOY {doy} to {output_file}")

print("Quantile mapping bias correction completed for all days of the year.")


In [ ]:
sst_selected = []

for year in range(1984, 2018):
    date = pd.date_range(start=f'{year}-01-01', end=f'{year}-12-31', freq='D')

    for day_of_year in range(len(date)):
            file_path = f'path/qm_adjusted_doy_{str(day_of_year+1).zfill(3)}.nc'
            ds = xr.open_dataset(file_path)

            if np.any(ds['time'] == date[day_of_year]):
                sst_day = ds.sst.sel(time=date[day_of_year])
                sst_selected.append(sst_day)

len(sst_selected)

In [ ]:
sst_full_xr = xr.concat(sst_selected, dim='time')
sst_full_xr.shape

In [ ]:
sst_full_xr.to_netcdf('path/bias_corrected_quantile_CanESM5_daily_sst_1984_2017.nc', 'w')